# Building Constructors Dimension


In [0]:
dbutils.widgets.text("p_batch_id", "")
v_batch_id = dbutils.widgets.get("p_batch_id")

In [0]:
 %run ../00-common/1.environment_config

In [0]:
%run ../00-common/4.gold_helpers

In [0]:
constructors_table = f"{catalog_name}.{silver_schema}.constructors"
ref_nationality_region_table = f"{catalog_name}.{gold_schema}.ref_nationality_region"
# target_table
target_table = f"{catalog_name}.{gold_schema}.dim_constructors"

## Reading constructors table and gold.ref_nationality_region

In [0]:
constructors_df = (
    spark.table(constructors_table)
    .filter(F.col("batch_id") == v_batch_id)
)
ref_nationality_region_df = spark.table(ref_nationality_region_table)

## Joining the 2 tables

In [0]:
dim_constructors_df = (
    constructors_df
    .join(
        ref_nationality_region_df,
        constructors_df.nationality == ref_nationality_region_df.nationality,
        how = "left"
    )
    .select(
        constructors_df.constructor_id,
        constructors_df.constructor_name,
        constructors_df.nationality,
        ref_nationality_region_df.region.alias("nationality_region")
    )
)

In [0]:
display(dim_constructors_df)

## Writing into the Gold Delta Table

In [0]:
dim_constructors_columns_to_update = [
    "constructor_name",
    "nationality",
    "nationality_region"
]

write_to_gold(
    source_df=dim_constructors_df,
    target_table=target_table,
    merge_condition= "t.constructor_id = s.constructor_id",
    columns_to_update = dim_constructors_columns_to_update
)

In [0]:
spark.table(target_table).display()